## Load ResNet18

This is the most important idea:

ResNet18 was originally trained on many general images
you reuse that knowledge
then you replace the last layer so it predicts your emotion labels

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

num_classes = 5  # angry, happy, neutral, sad, surprise

# Load pretrained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Replace final classification layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

print(model)

## Prepare the images

ResNet18 expects images in a standard format. So we resize and normalize them.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_data = datasets.ImageFolder("dataset/train", transform=train_transform)
val_data = datasets.ImageFolder("dataset/val", transform=val_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

print(train_data.classes)

##Choose loss function and optimizer

Meaning
CrossEntropyLoss is used for classification
Adam updates the model weights during training

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Train the model
For each batch of images:

model looks at images
predicts emotion
compares prediction with correct label
learns from the mistake
improves little by little

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()           # clear old gradients
        outputs = model(images)         # forward pass
        loss = criterion(outputs, labels)
        loss.backward()                 # backpropagation
        optimizer.step()                # update weights

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss:.4f}, Accuracy: {acc:.2f}%")

## Validate the model

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_acc = 100 * correct / total
print(f"Validation Accuracy: {val_acc:.2f}%")

## Save the trained model

In [ ]:
torch.save(model.state_dict(), "resnet18_emotion.pth")

## Load the model again later

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

num_classes = 5

model = models.resnet18(weights=None)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

model.load_state_dict(torch.load("resnet18_emotion.pth", map_location="cpu"))
model.eval()

## Predict one new image

In [ ]:
from PIL import Image

class_names = ['angry', 'happy', 'neutral', 'sad', 'surprise']

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

image = Image.open("test_face.jpg").convert("RGB")
image = transform(image).unsqueeze(0)   # add batch dimension

with torch.no_grad():
    outputs = model(image)
    _, predicted = torch.max(outputs, 1)

print("Predicted emotion:", class_names[predicted.item()])

## Predict one new image

Why unsqueeze(0)?

The model expects a batch of images, not just one image.
So this changes shape from:

In [ ]:
from PIL import Image

class_names = ['angry', 'happy', 'neutral', 'sad', 'surprise']

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

image = Image.open("test_face.jpg").convert("RGB")
image = transform(image).unsqueeze(0)   # add batch dimension

with torch.no_grad():
    outputs = model(image)
    _, predicted = torch.max(outputs, 1)

print("Predicted emotion:", class_names[predicted.item()])